From Ready to first 5x5x5 box: 24 / -17 / -5

Box1: 0.135 / 0.24 \
Box2: 0.185 / 0.17

In [1]:
import rby1_sdk as rby
from dynamixel_sdk import PortHandler, PacketHandler
import time
import threading
import numpy as np
import functions as fn

np.set_printoptions(suppress=True, precision=5, linewidth=300)  

SIM_OR_REAL = 1         # 0: Simulation / 1: Real Robot(Wifi mode) / 2: Real Robot (UPC mode)


In [2]:
if (SIM_OR_REAL == 2):
    # --- Gripper ------------------------------------ # 
    PROTOCOL_VERSION = 2.0
    BAUDRATE = 2000000
    DEVICE_NAME_GRIPPER = "/dev/rby1_gripper"

    ADDR_OPERATING_MODE     = 11                # 0: Current Control / 1: Velocity Control / 3: Position Control(default)
    ADDR_TORQUE_ENABLE      = 64                # 0: Turn OFF(default) / 1: Turn ON
    ADDR_LED                = 65                # 0: Turn OFF(default) / 1: Turn ON
    ADDR_GOAL_CURRENT       = 102
    ADDR_GOAL_POSITION      = 116   
    ADDR_PRESENT_CURRENT    = 126               # Unit: 2.69mA
    ADDR_PRESENT_POSITION   = 132               # Unit: 1-pulse(or Tick)

    CURRENT_CONTROL_MODE    = 0
    POSTION_CONTROL_MODE    = 3
    TORQUE_DISABLE          = 0
    TORQUE_ENABLE           = 1

    DXL_MIN_POS_VALUE       = 0
    DXL_MAX_POS_VALUE       = 4095

    RESOLUTION_TICKS_PER_REV    = 4096.0
    TWO_PI                      = 2.0 * np.pi
    RAD2DEG                     = 180.0 / np.pi

    R_DXL_ID                    = 0
    L_DXL_ID                    = 1

    is_init_finish              = False

In [3]:
if (SIM_OR_REAL == 0):
    ROBOT_ADDRESS = "localhost:50051"           # For Docker Simulation
elif (SIM_OR_REAL == 1 or SIM_OR_REAL == 2):
    ROBOT_ADDRESS = "192.168.30.1:50051"        # For Real Robot
else:
    ROBOT_ADDRESS = "localhost:50051"           # For Safety, Default=Docker Simulation

robot = rby.create_robot_a(ROBOT_ADDRESS)   # Robot Model is "A"
model = robot.model()
if not robot.connect():
    print("Could not connect to robot")
    exit(1)
else:
    print("SUCCESS: Connect to the robot.")
    if (robot.power_on(".*") == True):         # Turn on all the power (5v, 12v, 24v, 48v)
        time.sleep(1)
        if (robot.servo_on(".*") == True):         # Enable servo motors on all actuators
            time.sleep(1)
            robot.reset_fault_control_manager()
            robot.enable_control_manager(unlimited_mode_enabled=True)
            print("SUCCESS: Power ON the Robot.")
            
            # 여기 추가: 그리퍼 쪽 12V 출력 ON
            time.sleep(1)
            robot.set_tool_flange_output_voltage("right", 12)
            robot.set_tool_flange_output_voltage("left", 12)
            print("SUCCESS: Power ON the Gripper.")
        else:
            print("Failed to power ON the Servo")

SUCCESS: Connect to the robot.
SUCCESS: Power ON the Robot.
SUCCESS: Power ON the Gripper.


In [ ]:
if (SIM_OR_REAL == 2):
    # --- Open Gripper ports ---
    port_hdlr = PortHandler(DEVICE_NAME_GRIPPER)
    packet_hdlr = PacketHandler(PROTOCOL_VERSION)

    if port_hdlr.openPort():
        print(f"SUCCESS: open the port: {DEVICE_NAME_GRIPPER}")
    else:
        raise RuntimeError(f"Failed to open the port: {DEVICE_NAME_GRIPPER}")

    if port_hdlr.setBaudRate(BAUDRATE):
        print(f"SUCCESS: change the baudrate {BAUDRATE}")
    else:
        raise RuntimeError(f"Failed to change the baudrate")
    
    # --- Ping Test ---
    dxl_activated = []

    for dxl_id in [R_DXL_ID, L_DXL_ID]:
        dxl_model_num, dxl_comm_res, dxl_err = packet_hdlr.ping(port_hdlr, dxl_id)
        
        if dxl_comm_res != fn.COMM_SUCCESS:
            print("[DXL ID:%d] %s" % (dxl_id, packet_hdlr.getTxRxResult(dxl_comm_res)))
        elif dxl_err != 0:
            print("[DXL ID:%d] %s" % (dxl_id, packet_hdlr.getRxPacketError(dxl_err)))
        else:
            dxl_activated.append(dxl_id)
            print("[DXL ID:%d] ping Succeeded. Dynamixel model number : %d" % (dxl_id, dxl_model_num))
    
    # --- Enable Current Control ---
    for dxl_id in dxl_activated:
        fn.DXL_write_torque_mode(dxl_id, TORQUE_DISABLE, port_hdlr, packet_hdlr)         # Operation mode 변경 전 torque OFF
        fn.DXL_write_operation_mode(dxl_id, CURRENT_CONTROL_MODE, port_hdlr, packet_hdlr)
        fn.DXL_write_torque_mode(dxl_id, TORQUE_ENABLE, port_hdlr, packet_hdlr)

In [5]:
# robot.set_tool_flange_output_voltage("right", 0)
# robot.set_tool_flange_output_voltage("left", 0)
# time.sleep(1)
# robot.disable_control_manager()
# robot.power_off(".*")

In [6]:
# Desired_Pos = fn.PACK_POS      # ZERO_POS, READY_POS, PACK_POS
# fn.move_position(robot, Desired_Pos, minimum_time=10)

In [4]:
Desired_Pos = fn.READY_DEMO_POS      # ZERO_POS, READY_POS, PACK_POS, READY_DEMO_POS
fn.move_position(robot, Desired_Pos, minimum_time=3)

True

In [8]:
# right_deltas = [
#     [0, -0.1, 0],
#     [0, 0.1, 0],
#     [0, 0, 0.1],
#     [0, 0, -0.1]
# ]

# stream = robot.create_command_stream()
# for delta in right_deltas:
#     stream.send_command(fn.move_endeffector(robot, fn.RIGHT_END, delta))
#     time.sleep(2.0)
# stream.cancel()
# stream.wait()

In [9]:
# pos_delta = [0, 0.0, 0.1]
# fn.move_arm_pos(robot, fn.LEFT_END, pos_delta)

In [10]:
# pos_delta = [0.24, -0.17, 0.1]
# rpy_delta = [0.0, 0.0, np.deg2rad(90.0)]
# fn.move_arm_6D_relpos_relori(robot, fn.LEFT_END, pos_delta, rpy_delta)

In [11]:
# pos_delta = [0.1, -0.1, 0.05]
# pos_delta = [0, 0, 0]
# rpy_from_torso = [0, 0, 0]
# fn.move_arm_6D_relpos_globalori(robot, fn.LEFT_END, pos_delta, rpy_from_torso)

In [12]:
# pos_delta = [0.2, 0, 0]
# fn.move_arm_relpos(robot, fn.LEFT_END, pos_delta)

In [13]:
# # 0.2 / -0.06 / 
# pos_delta_from_torso = [0.23, -0.06, 0.0]
# fn.move_arm_globalpos(robot, fn.LEFT_END, pos_delta_from_torso)

In [5]:
pos_delta_from_torso_R = [0.15, 0.14, 0.0]
pos_delta_from_torso_L = [0.15, -0.14, 0.0]
fn.move_arm_globalpos_both(robot, pos_delta_from_torso_R, pos_delta_from_torso_L)

T_target_R
: [[ 0.00001 -0.34201 -0.9397   0.51553]
 [ 0.99619 -0.08191  0.02983 -0.09554]
 [-0.08717 -0.93612  0.3407  -0.09757]
 [ 0.       0.       0.       1.     ]]
T_target_L
: [[ 0.       0.342   -0.9397   0.51553]
 [-0.9962  -0.08189 -0.0298   0.09554]
 [-0.08715  0.93613  0.34069 -0.09756]
 [ 0.       0.       0.       1.     ]]


True

In [ ]:
# x = -0.03
# pos_delta_from_torso_R = [x, 0.0, 0.0]
# pos_delta_from_torso_L = [x, 0.0, 0.0]
# fn.move_arm_globalpos_both(robot, pos_delta_from_torso_R, pos_delta_from_torso_L)

T_target_R
: [[-0.00005 -0.34197 -0.93971  0.51552]
 [ 0.99622 -0.08166  0.02966 -0.09555]
 [-0.08688 -0.93616  0.34068 -0.09757]
 [ 0.       0.       0.       1.     ]]
T_target_L
: [[-0.00005  0.34186 -0.93975  0.51552]
 [-0.99623 -0.08155 -0.02961  0.09554]
 [-0.08676  0.9362   0.34058 -0.09755]
 [ 0.       0.       0.       1.     ]]


True

In [6]:
z = -0.1
pos_delta_from_torso_R = [0.0, 0.0, z]
pos_delta_from_torso_L = [0.0, 0.0, z]
fn.move_arm_globalpos_both(robot, pos_delta_from_torso_R, pos_delta_from_torso_L)

T_target_R
: [[ 0.00001 -0.34199 -0.9397   0.51553]
 [ 0.99619 -0.08193  0.02983 -0.09554]
 [-0.08719 -0.93613  0.34069 -0.19757]
 [ 0.       0.       0.       1.     ]]
T_target_L
: [[-0.00002  0.34197 -0.93971  0.51553]
 [-0.99619 -0.08196 -0.02981  0.09554]
 [-0.08721  0.93613  0.34067 -0.19756]
 [ 0.       0.       0.       1.     ]]


True

In [ ]:
fn.DXL_write_current_ampere(0, 0.03, port_hdlr, packet_hdlr)
fn.DXL_write_current_ampere(1, 0.03, port_hdlr, packet_hdlr)

In [15]:
# pos_delta_from_torso_R = [0.0, 0.0, -0.13]
# pos_delta_from_torso_L = [0.0, 0.0, -0.13]
# fn.move_arm_globalpos_both(robot, pos_delta_from_torso_R, pos_delta_from_torso_L)

In [16]:
# rpy_from_torso = [0, 0, 0]
# fn.move_arm_globalori(robot, fn.LEFT_END, rpy_from_torso)